In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Smart MCQ Solver — Data Loading & EDA
Load train data and inspect basic structure before any preprocessing.

In [ ]:
df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

In [ ]:
df.shape

In [ ]:
df.head(3)

#### Checking Null Values: 0 

In [ ]:
df.isnull().sum()

#### Drop unused ID column

In [ ]:
df=df.drop('id',axis=1)

## Duplicate Check
Checking duplicate prompts — kept intentionally (empirically improved leaderboard score vs. dropping them).

In [ ]:
df.duplicated(subset='prompt').sum()

## Basic Feature Engineering: Length & Word Overlap
Prompt/option character lengths and prompt-option word overlap counts, used later for baseline heuristics and the ANN. Added directly to the entire train set as these features dont cause data leakage.

In [ ]:
df['prompt_len']=df['prompt'].str.len()
for col in ['A','B','C','D','E']:
    df[f'{col}_len']=df[col].str.len()

def get_overlaps(row):
    prompt_words=set(row['prompt'].split())
    overlaps={}
    for col in ['A','B','C','D','E']:
        option_words=set(row[col].split())
        overlaps[f'{col}_overlap']=len(prompt_words & option_words)
    return pd.Series(overlaps)

overlaps_df=df.apply(get_overlaps,axis=1)
df=pd.concat([df,overlaps_df],axis=1)

## EDA: Answer Distribution
Checking class balance across A-E. Answers are slightly biased towards option B and least biased with option D.

In [ ]:
import matplotlib.pyplot as plt

answer=df['answer']
answer.value_counts().sort_index().plot(kind='pie')
plt.xlabel('Answer')
plt.ylabel('Count')
plt.title('Answer Distribution')

## EDA: Length Bias Check
Testing whether the correct answer is systematically the longest option.

In [ ]:
len_cols=['A_len','B_len','C_len','D_len','E_len']
lengths=df[len_cols].values
rank_of_correct=[]

for i,ans in enumerate(df['answer']):
    idx="ABCDE".index(ans)
    row_lengths=lengths[i]
    rank=(row_lengths > row_lengths[idx]).sum()
    rank_of_correct.append(rank)

pd.Series(rank_of_correct).value_counts(normalize=True).sort_index().plot(kind='bar')
plt.xlabel("Length rank of correct answer (0=longest, 4=shortest)")
plt.title("Is the correct answer usually the longest option?")
plt.show()

## EDA: Length Distributions
Sanity check on prompt/option length range — informs tokenizer max_length choice. So many of them have character length between 100 to 200. 

In [ ]:
plt.figure(figsize=(10,4))
plt.hist(df['prompt_len'],bins=30,alpha=0.7,label='prompt')
for col in ['A','B','C','D','E']:
    plt.hist(df[f'{col}_len'],bins=30,alpha=0.5,label=col)
plt.xlabel('Character length')
plt.ylabel('Count')
plt.title('Prompt & Option Length Distributions')
plt.legend()
plt.show()

## EDA: Word Cloud
Most frequent terms across all prompts (stopwords + MCQ-boilerplate filtered out). Based on the common words, it is clear that the questions might be based on Science with predominant weightage to Physics followed by Biological Sciences.

In [ ]:
from wordcloud import WordCloud,STOPWORDS
import matplotlib.pyplot as plt

text=" ".join(df['prompt'].tolist())
custom_stopwords=[
    'following', 'listed', 'choices', 'choice', 'options', 'option',
    'correct', 'answer', 'answers', 'statement', 'statements',
    'identify', 'select', 'determine', 'pick', 'choose',
    'context', 'given', 'carefully', 'possible', 'best',
    'among', 'accurately', 'describes', 'describe', 'accurate',
    'based', 'relationship', 'role', 'reason', 'significance',
    'associated', 'used', 'term', 'distinction', 'defined', 'definition',
    'concept', 'measuring', 'origin', 'proposed', 'purpose',
    'focus', 'main', 'formation', 'processes'
]
wc=WordCloud(width=2000,height=1000,background_color='white',scale=2,stopwords=STOPWORDS.union(custom_stopwords),max_words=200).generate(text)

plt.figure(figsize=(12,6))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title("Most common words in all prompts")
plt.show()

Separate Target Variable

In [ ]:
answer=df['answer']
df=df.drop('answer',axis=1)

## Train/Validation with Group Shuffle Split
80/20 split, no stratification, fixed random_state.

In [ ]:
def option_key(row,cols=['A','B','C','D','E']):
    return frozenset(str(row[c]).strip().lower() for c in cols)

df['group_key']=df.apply(option_key,axis=1)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss=GroupShuffleSplit(n_splits=1,train_size=0.8,random_state=1)
train_idx,val_idx=next(gss.split(df,groups=df['group_key']))
x_train,x_val=df.iloc[train_idx].reset_index(drop=True),df.iloc[val_idx].reset_index(drop=True)
y_train,y_val=answer.iloc[train_idx].reset_index(drop=True),answer.iloc[val_idx].reset_index(drop=True)

In [ ]:
x_train=x_train.drop(columns=['group_key'])
x_val=x_val.drop(columns=['group_key'])

### Helper Functions : MAP@3, Other Scores and Scores to Labels mapping

In [ ]:
def score_to_labels(scores,top1=False):
    labels=[]
    top1_labels=[]
    for score in scores:
        scored=np.argsort(score)[::-1][:3]
        top_3=[chr(65+i) for i in scored]
        labels.append(" ".join(top_3))
        top1_labels.append(chr(65+np.argsort(score)[::-1][0]))
    if not top1:
        return labels
    else:
        return top1_labels

In [ ]:
def map_at_3(predictions,ground_truth):
    ap_scores=[]
    for pred, truth in zip(predictions, ground_truth):
        pred_list = pred.split()
        if truth in pred_list:
            ap_scores.append(1.0 / (pred_list.index(truth) + 1))
        else:
            ap_scores.append(0.0)
    return sum(ap_scores) / len(ap_scores)

In [ ]:
from sklearn.metrics import accuracy_score,f1_score

def support_metrics(train_scores,val_scores):
    train_acc=accuracy_score(score_to_labels(train_scores,top1=True),y_train)
    val_acc=accuracy_score(score_to_labels(val_scores,top1=True),y_val)
    train_f1=f1_score(score_to_labels(train_scores,top1=True),y_train,average='macro')
    val_f1=f1_score(score_to_labels(val_scores,top1=True),y_val,average='macro')
    return train_acc,val_acc,train_f1,val_f1

## Additional Features: Special-Option Flags & Length Rank
Flagging "none/all of the above"-style options, and ranking options by length per row.

In [ ]:
def special_option(text):
    text=str(text).lower()
    patterns=['none of the above','all of the above','none of these','all of these','both a and b','both a and c','both b and c','not applicable']
    return int(any(p in text for p in patterns))

In [ ]:
for col in ['A','B','C','D','E']:
    x_train[f'{col}_is_special']=x_train[col].apply(special_option)
for col in ['A','B','C','D','E']:
    x_val[f'{col}_is_special']=x_val[col].apply(special_option)
x_train.head(12)

In [ ]:
def length_rank_features(row):
    option_cols=['A','B','C','D','E']
    lengths={col:row[f'{col}_len'] for col in option_cols}
    sorted_options=sorted(option_cols,key=lambda col:lengths[col],reverse=True)
    ranks={}
    for rank,col in enumerate(sorted_options):
        ranks[f'{col}_len_rank']=rank
    return pd.Series(ranks)

rank_df_train=x_train.apply(length_rank_features,axis=1)
x_train=pd.concat([x_train,rank_df_train],axis=1)

rank_df_val=x_val.apply(length_rank_features,axis=1)
x_val=pd.concat([x_val,rank_df_val],axis=1)

## Experiment Tracking Setup (W&B)

In [ ]:
!pip install wandb -q

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient
import os

secrets=UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
wandb.login()
wandb_mode=True

# Baseline 1: Longest-Option Heuristic
Naive baseline — always guess the longest option first.

In [ ]:
def len_scores(row):
    lengths=[row['A_len'],row['B_len'],row['C_len'],row['D_len'],row['E_len']]
    max_len=max(lengths)
    min_len=min(lengths)
    scores=[]
    for length in lengths:
        score=0
        if length==max_len:
            score=0.5
        elif length==min_len:
            score=0.2
        else:
            score=0.3
        scores.append(score)
    return scores

length_train_scores=x_train.apply(len_scores,axis=1).to_list()
length_labels=score_to_labels(length_train_scores)
length_map3=map_at_3(length_labels,y_train)
print(length_map3)

In [ ]:
length_val_scores=x_val.apply(len_scores,axis=1).to_list()
length_val_labels=score_to_labels(length_val_scores)
length_val_map3=map_at_3(length_val_labels,y_val)
length_val_map3

In [ ]:
from sklearn.metrics import accuracy_score,f1_score

len_train_acc,len_val_acc,len_train_f1,len_val_f1=support_metrics(length_train_scores,length_val_scores)

In [ ]:
if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="largestlen-baseline-metrics",id="largelen_v1",resume="allow")
    wandb.log({
        "map_at_3_train":length_map3,   
        "map_at_3_val":length_val_map3,
        "accuracy_at_train":len_train_acc,
        "accuracy_at_val":len_val_acc,
        "f1_at_train":len_train_f1,
        "f1_at_val":len_val_f1,
        "metric":"Longest-Length"
        })
    wandb.finish()

# Baseline 2: TF-IDF Similarity
Scoring each option by prompt-option TF-IDF overlap.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf=TfidfVectorizer(sublinear_tf=True,ngram_range=(1,2))
all_text=x_train['prompt'].tolist()+x_train['A'].tolist()+x_train['B'].tolist()+x_train['C'].tolist()+x_train['D'].tolist()+x_train['E'].tolist()
tfidf.fit(all_text)

In [ ]:
tfidf_train_scores=[]
for col in ['A', 'B', 'C', 'D', 'E']:
    combined=(x_train['prompt']+" "+x_train[col]).to_list()
    scores=tfidf.transform(combined).sum(axis=1)
    tfidf_train_scores.append(np.asarray(scores).flatten())
tfidf_train_scores=np.array(tfidf_train_scores).T

In [ ]:
tfidf_val_scores=[]
for col in ['A', 'B', 'C', 'D', 'E']:
    combined=(x_val['prompt']+" "+x_val[col]).to_list()
    scores=tfidf.transform(combined).sum(axis=1)
    tfidf_val_scores.append(np.asarray(scores).flatten())
tfidf_val_scores=np.array(tfidf_val_scores).T

In [ ]:
tfidf_train_labels=score_to_labels(tfidf_train_scores,top1=False)
tfidf_train_map3=map_at_3(tfidf_train_labels,y_train)
print(tfidf_train_map3)

tfidf_val_labels=score_to_labels(tfidf_val_scores,top1=False)
tfidf_val_map3=map_at_3(tfidf_val_labels,y_val)
print(tfidf_val_map3)

In [ ]:
tfidf_train_acc,tfidf_val_acc,tfidf_train_f1,tfidf_val_f1=support_metrics(tfidf_train_scores,tfidf_val_scores)

In [ ]:
if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="tfidf-baseline-metrics",id="tfidf_v1",resume="allow")
    wandb.log({
        "map_at_3_train":tfidf_train_map3,   
        "map_at_3_val":tfidf_val_map3,
        "accuracy_at_train":tfidf_train_acc,
        "accuracy_at_val":tfidf_val_acc,
        "f1_at_train":tfidf_train_f1,
        "f1_at_val":tfidf_val_f1,
        "vectorizer":"TfidfVectorizer",
        "sublinear_tf":True})
    wandb.finish()

In [ ]:
import torch
import numpy as np
import random

seed=42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

# Baseline 3 : Word2Vec Embeddings
Training a lightweight Word2Vec model on train text for prompt-option cosine similarity scoring, feeding into the ANN's input features.

In [ ]:
from gensim.models import Word2Vec
np.random.seed(1)
from numpy.random import seed
seed(1)

sentences=[text.split() for text in all_text]
word2vec=Word2Vec(sentences,vector_size=10,window=1,min_count=1,seed=1,workers=1,epochs=10)

In [ ]:
option_cols=['A','B','C','D','E']
def sentence_embedding(text):
    words=text.split()
    embedding=[word2vec.wv[word] for word in words if word in word2vec.wv]
    if len(embedding)==0:
        return np.zeros(word2vec.vector_size)
    return np.mean(embedding,axis=0)

In [ ]:
prompts=np.vstack(x_train['prompt'].apply(sentence_embedding))
w2v_train_scores=[]

for col in option_cols:
    option_matrix=np.vstack(x_train[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(prompts,option_matrix))
    w2v_train_scores.append(scores)
w2v_train_scores=np.array(w2v_train_scores).T

In [ ]:
prompts=np.vstack(x_train['prompt'].apply(sentence_embedding))
w2v_train_scores=[]

for col in option_cols:
    option_matrix=np.vstack(x_train[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(prompts,option_matrix))
    w2v_train_scores.append(scores)
w2v_train_scores=np.array(w2v_train_scores).T

In [ ]:
val_prompts=np.vstack(x_val['prompt'].apply(sentence_embedding))
w2v_val_scores=[]

for col in option_cols:
    option_matrix=np.vstack(x_val[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(val_prompts,option_matrix))
    w2v_val_scores.append(scores)
w2v_val_scores=np.array(w2v_val_scores).T

In [ ]:
w2v_train_labels=score_to_labels(w2v_train_scores)
w2v_train_map3=map_at_3(w2v_train_labels,y_train)
print(w2v_train_map3)

w2v_val_labels=score_to_labels(w2v_val_scores)
w2v_val_map3=map_at_3(w2v_val_labels,y_val)
print(w2v_val_map3)

In [ ]:
w2v_train_acc,w2v_val_acc,w2v_train_f1,w2v_val_f1=support_metrics(w2v_train_scores,w2v_val_scores)

if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="word2vec-baseline-metrics",id="w2v_v1", resume="allow")
    wandb.log({
        "map_at_3_train":w2v_train_map3,   
        "map_at_3_val":w2v_val_map3,
        "accuracy_at_train":w2v_train_acc,
        "accuracy_at_val":w2v_val_acc,
        "f1_at_train":w2v_train_f1,
        "f1_at_val":w2v_val_f1,
        "embedding_model":"Word2Vec",
        "vector_size":20,
         "window":1,
         "min_count":1})
    wandb.finish()

In [ ]:
cols=x_train.columns
cols=cols.drop(['prompt',"A","B","C","D",'E'])

In [ ]:
len(cols)

#### Label Encoding

In [ ]:
label_map={"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

y_train_idx=y_train.map(label_map).values if y_train.dtype=='object' else y_train.values
y_val_idx=y_val.map(label_map).values if y_val.dtype=='object' else y_val.values

In [ ]:
from torch.utils.data import TensorDataset,DataLoader
from sklearn.preprocessing import StandardScaler

ss=StandardScaler()

x_train_t=torch.tensor(ss.fit_transform(np.concatenate((tfidf_train_scores,np.array(length_train_scores),w2v_train_scores,x_train[cols].values),axis=1)),dtype=torch.float32)
x_val_t=torch.tensor(ss.transform(np.concatenate((tfidf_val_scores,np.array(length_val_scores),w2v_val_scores,x_val[cols].values),axis=1)),dtype=torch.float32)
y_train_t=torch.tensor(y_train_idx,dtype=torch.long)
y_val_t=torch.tensor(y_val_idx,dtype=torch.long)

# Baseline 4 (MODEL 1): Custom ANN (MCQANN)
Small feed-forward network over TF-IDF + length + Word2Vec + other engineered features.

In [ ]:
from torch.utils.data import TensorDataset,DataLoader

train_ds=TensorDataset(x_train_t,y_train_t)
train_loader=DataLoader(train_ds,batch_size=16,shuffle=True)

In [ ]:
import torch.nn as nn

class MCQANN(nn.Module):
    def __init__(self,input_dim,num_choices=5,hidden_dim=32):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(hidden_dim,hidden_dim//2),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(hidden_dim//2,num_choices)
        )
    def forward(self,x):
        return self.net(x)

model=MCQANN(input_dim=x_train_t.shape[1])

In [ ]:
wandb.init(project="24f2002314-t22026",name="MCQ-ANN-scaled",id="ann_version_scaled2",resume="allow")
wandb.log({
        "lr":3e-4,   
        "weight_decay":1e-4,
        "epochs": 400,
        "patience": 15,
        "hidden_dimension":16,
        "batch_size": 32,
        "architecture": "MCQANN",
        "input_dim": x_train_t.shape[1],})

In [ ]:
import copy

criteria=nn.CrossEntropyLoss()
EPOCHS=400
optimizer=torch.optim.Adam(model.parameters(),lr=3e-4,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS,eta_min=1e-6)
device="cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

best_val_loss=float('inf')
patience=15
patience_count=0
best_val_map3=0

for epoch in range(EPOCHS):
    model.train()
    total_loss=0
    for x,y in train_loader:
        x,y=x.to(device),y.to(device)
        optimizer.zero_grad()
        output=model(x)
        loss=criteria(output,y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
        optimizer.step()
        total_loss+=loss.item()
    scheduler.step()

    if epoch%1==0:
        model.eval()
        with torch.no_grad():
            val_ds=x_val_t.to(device)
            val_out=model(val_ds)
            val_loss=criteria(val_out,y_val_t.to(device))
            val_acc=(val_out.argmax(dim=1).cpu()==y_val_t).float().mean()
            val_scores=val_out.cpu().numpy()
            val_labels=score_to_labels(val_scores,top1=False)
            val_map3=map_at_3(val_labels, y_val)
            avg_train_loss=total_loss/len(train_loader)
        print(f'Epoch :{epoch} - train loss : {avg_train_loss:.4f} val loss : {val_loss:.4f} val_acc={val_acc:.4f} val_map3={val_map3:.4f}')
        wandb.log({
            "epoch": epoch,
            "train_loss": avg_train_loss,
            "val_loss": val_loss.item(),
            "val_acc": val_acc.item(),
            "val_map3": val_map3
        })
            
    if val_map3>=best_val_map3+1e-4:
        best_val_loss=val_loss
        best_val_map3=val_map3
        patience_count=0
        best_state=copy.deepcopy(model.state_dict())
    else:
        patience_count+=1
        if patience_count>=patience:
            print(f"Early stopping at epoch {epoch}")
            break
model.load_state_dict(best_state)

In [ ]:
device='cuda' if torch.cuda.is_available() else 'cpu'
model.eval()
with torch.no_grad():
    predictions=model(x_val_t.to(device)).argmax(dim=1).cpu().numpy()
    targets=y_val_t.to(device).cpu().numpy()

accuracy_ann=accuracy_score(targets,predictions)
print(f"Accuracy: {accuracy_ann}")

In [ ]:
wandb.run.summary["best_val_loss"]=best_val_loss.item()
wandb.run.summary["best_val_map3"]=val_map3   
wandb.run.summary["stopped_epoch"]=epoch
torch.save(best_state,"best_model.pt")
artifact=wandb.Artifact("mcq_ann_best_model",type="model")
artifact.add_file("best_model.pt")
wandb.log_artifact(artifact)
wandb.finish()

### Building the dataset

In [ ]:
from datasets import Dataset

label_map={'A':0,'B':1,'C':2,'D':3,'E':4}

x_train['label']=y_train.map(label_map)
x_val['label']=y_val.map(label_map)
train_dataset=Dataset.from_pandas(x_train.reset_index(drop=True))
val_dataset=Dataset.from_pandas(x_val.reset_index(drop=True))

### Helper Function for Option Augmentation/Shuffling

In [ ]:
import random

def shuffle_option(row):
    cols=['A','B','C','D','E']
    correct_idx=row['label']
    correct_ans=row[cols[correct_idx]]
    values=[row[c] for c in cols]
    random.shuffle(values)
    new=values.index(correct_ans)
    for i,c in enumerate(cols):
        row[c]=values[i]
    row['label']=new
    return row

In [ ]:
train_dataset=train_dataset.map(shuffle_option)

# Transformer Fine-Tuning (Pretrained Models 2 and 3: Electra & RoBERTa)
Multiple-choice fine-tuning on the augmented training set (duplicates kept, options shuffled/relabeled). This is the core model for final predictions.

In [ ]:
from transformers import AutoModelForMultipleChoice,AutoTokenizer,TrainingArguments,Trainer
import torch

model_name1='google/electra-large-discriminator'
tokeniser1=AutoTokenizer.from_pretrained(model_name1)
model1=AutoModelForMultipleChoice.from_pretrained(model_name1,ignore_mismatched_sizes=True)

model_name2='roberta-base'
tokeniser2=AutoTokenizer.from_pretrained(model_name2)
model2=AutoModelForMultipleChoice.from_pretrained(model_name2,ignore_mismatched_sizes=True)

def preprocessing(data,tokeniser):
    prompts=[[p]*5 for p in data['prompt']]
    options=[[data[o][i] for o in ['A','B','C','D','E']] for i in range(len(data['prompt']))]
    prompts=sum(prompts,[])
    options=sum(options,[])
    processed=tokeniser(prompts,options,truncation=True,max_length=256,padding='max_length')
    result={k:[v[i:i+5] for i in range(0,len(v),5)] for k,v in processed.items()}
    if "label" in data:
         result["label"]=data["label"]

    return result

processed_train1=train_dataset.map(lambda x:preprocessing(x,tokeniser1),batched=True,remove_columns=train_dataset.column_names)
processed_val1=val_dataset.map(lambda x:preprocessing(x,tokeniser1),batched=True,remove_columns=val_dataset.column_names)
processed_train2=train_dataset.map(lambda x:preprocessing(x,tokeniser2),batched=True,remove_columns=train_dataset.column_names)
processed_val2=val_dataset.map(lambda x:preprocessing(x,tokeniser2),batched=True,remove_columns=val_dataset.column_names)

In [ ]:
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits,labels=eval_pred
    top3=np.argsort(-logits,axis=1)[:,:3]
    ap=[1.0/(list(row).index(l)+1) if l in row else 0.0 for row,l in zip(top3,labels)]
    acc=(np.argmax(logits, axis=1)==labels).mean()
    preds=np.argmax(logits,axis=1)
    f1_macro=f1_score(labels,preds,average="macro")
    return {"map_at_3":np.mean(ap),"accuracy":acc,"f1_macro":f1_macro}

### Training Loop

In [ ]:
processed_train1.set_format(type="torch",columns=["input_ids","attention_mask","label"])
processed_val1.set_format(type="torch",columns=["input_ids","attention_mask","label"])

train_args1=TrainingArguments(output_dir="./electra-large-mcq",learning_rate=1e-5,eval_strategy="epoch",gradient_checkpointing=False,save_strategy="epoch",num_train_epochs=5,warmup_steps=100,remove_unused_columns=False,max_grad_norm=1.0,
                             fp16=False,weight_decay=0.01,logging_steps=10,metric_for_best_model="map_at_3",per_device_train_batch_size=2,gradient_accumulation_steps=4,per_device_eval_batch_size=2,load_best_model_at_end=True)

trainer1=Trainer(model=model1,args=train_args1,train_dataset=processed_train1,eval_dataset=processed_val1,compute_metrics=compute_metrics)
trainer1.train()

In [ ]:
processed_train2.set_format(type="torch",columns=["input_ids","attention_mask","label"])
processed_val2.set_format(type="torch",columns=["input_ids","attention_mask","label"])

train_args2=TrainingArguments(output_dir="./roberta-base-mcq",learning_rate=3e-5,eval_strategy="epoch",gradient_checkpointing=False,save_strategy="epoch",num_train_epochs=5,warmup_steps=500,remove_unused_columns=False,max_grad_norm=1.0,
                             fp16=False,weight_decay=0.01,logging_steps=10,metric_for_best_model="map_at_3",per_device_train_batch_size=8,per_device_eval_batch_size=8,load_best_model_at_end=True)

trainer2=Trainer(model=model2,args=train_args2,train_dataset=processed_train2,eval_dataset=processed_val2,compute_metrics=compute_metrics)
trainer2.train()

### Validation Predictions & Logging

In [ ]:
val_results1=trainer1.predict(processed_val1)
val_results2=trainer2.predict(processed_val2)

In [ ]:
if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="electra-large-final",id="electra-large-final",resume="allow")
    wandb.log({   
        "metrics":val_results1.metrics,
        "MODEL":"ELECTRA-large-DISCRIMINATOR",
        "epochs":5,
        "learning_rate":1e-5,
        "warm-up":300,
        "weight_decay":0.01
         })
    wandb.finish()

In [ ]:
if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="roberta-final",id="roberta-base-final",resume="allow")
    wandb.log({   
        "metrics":val_results2.metrics,
        "MODEL":"ROBERTA-BASE",
        "epochs":5,
        "learning_rate":3e-5,
        "warm-up":500,
        "weight_decay":0.01
         })
    wandb.finish()

## Obtained Results on Various Baselines

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Longest-Length', 'TF-IDF', 'Word2Vec', 'Custom ANN', 'Electra-large','Roberta-base'],
    'Val MAP@3': [length_val_map3, tfidf_val_map3, w2v_val_map3, best_val_map3, 
                  val_results1.metrics['test_map_at_3'],val_results2.metrics['test_map_at_3']],
    'Val Accuracy': [len_val_acc, tfidf_val_acc, w2v_val_acc,accuracy_ann,
                     val_results1.metrics['test_accuracy'],val_results2.metrics['test_accuracy']]
})
comparison

## Test Dataset Preprocessing and creating inference of the Selected Model - Electra LARGE

In [ ]:
test=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
test['prompt_len']=test['prompt'].str.len()
for col in ['A','B','C','D','E']:
    test[f'{col}_len']=test[col].str.len()

def get_overlaps(row):
    prompt_words=set(row['prompt'].split())
    overlaps={}
    for col in ['A','B','C','D','E']:
        option_words=set(row[col].split())
        overlaps[f'{col}_overlap']=len(prompt_words & option_words)
    return pd.Series(overlaps)

overlaps_test=test.apply(get_overlaps,axis=1)
test=pd.concat([test,overlaps_test],axis=1)

test_dataset=Dataset.from_pandas(test)
processed_test1=test_dataset.map(lambda x:preprocessing(x,tokeniser1),batched=True,remove_columns=test_dataset.column_names)
processed_test2=test_dataset.map(lambda x:preprocessing(x,tokeniser2),batched=True,remove_columns=test_dataset.column_names)

In [ ]:
test_results1=trainer1.predict(processed_test1)
logits1=test_results1.predictions
letters=np.array(["A","B","C","D","E"])
top3=np.argsort(logits1,axis=1)[:, ::-1][:, :3]
top3=[" ".join(letters[row]) for row in top3]
print(top3[:10])

In [ ]:
submission=pd.DataFrame({'id':test['id'],'Prediction':top3})
submission.to_csv('submission.csv',index=False)
print("Model submission saved!")
print(submission.head())